# DataStax DB

In [3]:
### Config
import os
from dotenv import load_dotenv

load_dotenv()
astra_db_api_endpoint = os.getenv("ASTRA_DB_API_ENDPOINT")
astra_db_token = os.getenv("ASTRA_DB_TOKEN")


In [4]:
from langchain_openai import OpenAIEmbeddings
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")
embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small",
    dimensions = 1024,
    api_key = api_key,
    base_url = base_url
)
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x0000024226A3C590>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x0000024226A3D010>, model='text-embedding-3-small', dimensions=1024, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base='https://api.chatanywhere.org', openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [5]:
from langchain_astradb import AstraDBVectorStore

vector_store = AstraDBVectorStore(
    embedding = embeddings,
    api_endpoint = astra_db_api_endpoint,
    token = astra_db_token,
    collection_name = "astra_vector_langchain",
    namespace = None
)
vector_store

In [6]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [7]:
vector_store.add_documents(
    documents = documents
)

['38f74d6f9a954f4d882b2b1ec3ebc75f',
 '6f3fb0df517b4b2495fa0c9920860606',
 '1207ba2bdbfc421da32e9994b921b378',
 'b13d991302e44b4586c8979e0066e8a8',
 '304a69cc8a8041a0b92edf6289d0ed15',
 '524ed39bb91f417ca5cb4e33b7cf4f3a',
 'b865c4a29a2242e4ba34a577546aa540',
 'ece9809d258c44239459f88bd767b1e8',
 '56dbd22b0c264f65b6c12e8815e7ea1e',
 '0e1513d2ae444a2ca72a1f4fb03bcbf9']

In [8]:
# Search from Vector Store DB

vector_store.similarity_search("What is the weather?")

[Document(id='6f3fb0df517b4b2495fa0c9920860606', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='0e1513d2ae444a2ca72a1f4fb03bcbf9', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
 Document(id='56dbd22b0c264f65b6c12e8815e7ea1e', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='b865c4a29a2242e4ba34a577546aa540', metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.')]

In [9]:
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k = 3,
    filter = {"source": "tweet"}
)
for res in results:
    print(f"* {res.page_content}, metadata: {res.metadata}")

* Building an exciting new project with LangChain - come check it out!, metadata: {'source': 'tweet'}
* LangGraph is the best framework for building stateful, agentic applications!, metadata: {'source': 'tweet'}
* Wow! That was an amazing movie. I can't wait to see it again., metadata: {'source': 'tweet'}


In [10]:
results = vector_store.similarity_search_with_score(
    "LangChain provides abstractions to make working with LLMs easy",
    k = 3,
    filter = {"source": "tweet"}
)
for res, score in results:
    print(f"* [SIM={score:.2f}] {res.page_content}, metadata: {res.metadata}")

* [SIM=0.72] Building an exciting new project with LangChain - come check it out!, metadata: {'source': 'tweet'}
* [SIM=0.71] LangGraph is the best framework for building stateful, agentic applications!, metadata: {'source': 'tweet'}
* [SIM=0.53] Wow! That was an amazing movie. I can't wait to see it again., metadata: {'source': 'tweet'}


In [14]:
# Retriever
retriever = vector_store.as_retriever(
    search_type = "similarity_score_threshold",
    search_kwargs = {"k": 2, "score_threshold": 0.6}
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='b13d991302e44b4586c8979e0066e8a8', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

In [16]:
# Retriever
retriever = vector_store.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k": 2}
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='b13d991302e44b4586c8979e0066e8a8', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(id='56dbd22b0c264f65b6c12e8815e7ea1e', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.')]